# 1. Scanning with SARUS (Mono + di PWMs)

In [ ]:
import os
import re
import sys
import argparse
import subprocess
import matplotlib
import time
import random
import string
import shlex
import shutil
import glob
import pickle
import csv
import operator
import joblib
import pandas as pd
import numpy as np
from itertools import groupby
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score, average_precision_score, accuracy_score
from sklearn.metrics import roc_curve
from sklearn.metrics import auc
from sklearn.metrics import precision_recall_curve
from sklearn.decomposition import PCA
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import cross_val_score
from sklearn.model_selection import GridSearchCV
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, f1_score, precision_score, recall_score, classification_report, confusion_matrix
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline
from sklearn.metrics import accuracy_score, confusion_matrix
from collections import defaultdict
from sklearn.ensemble import BaggingClassifier
from sklearn import svm
from sklearn.model_selection import StratifiedKFold
from Bio import SeqIO
from sklearn.feature_selection import RFE
from sklearn import preprocessing
import pybedtools as pbt
import pyBigWig as pbw
from datetime import date
from xgboost import XGBClassifier
from sklearn.model_selection import KFold
from sklearn.metrics import roc_auc_score
from itertools import chain
from Bio.Seq import Seq
from Bio.SeqRecord import SeqRecord
from sklearn.ensemble import AdaBoostClassifier
from adjustText import adjust_text
import matplotlib.ticker as ticker
import matplotlib.pyplot as plt
from pylab import *
%matplotlib inline

In [ ]:
#os.chdir("/home/pavelkrav/")

root = os.getcwd()
basicdir = os.path.abspath('Release/TF-ML/')
outputdir = os.path.abspath('Release/TF-ML/outputdir')
train_dir = os.path.abspath('Release/TF-ML/train/') 
test_dir = os.path.abspath('Release/TF-ML/test/') 

if not os.path.exists(basicdir):
    os.makedirs(basicdir)
if not os.path.exists(outputdir):
    os.makedirs(outputdir)
if not os.path.exists(train_dir):
    os.makedirs(train_dir)
if not os.path.exists(test_dir):
    os.makedirs(test_dir)


pwmdir_mono = os.path.abspath('./hocomoco11/models/pwm/mono/all')
pwmdir_di = os.path.abspath('./hocomoco11/models/pwm/di/all')


files_pwm_HUMAN_mono = []
files_pwm_MOUSE_mono = []

for d in os.listdir(pwmdir_mono):
    d = pwmdir_mono + "/" + d 
    if os.path.isdir(d):
        os.chdir(d)
        if d.split("_")[-1] == "HUMAN":
            files_pwm_HUMAN_mono.append([f for f in os.listdir(os.getcwd()) if os.path.splitext(f)[1] == '.pwm']) 
        if d.split("_")[-1] == "MOUSE":
            files_pwm_MOUSE_mono.append([f for f in os.listdir(os.getcwd()) if os.path.splitext(f)[1] == '.pwm'])
    os.chdir(pwmdir_mono)

print(len(files_pwm_HUMAN_mono))
print(len(files_pwm_MOUSE_mono))


files_pwm_HUMAN_di = []
files_pwm_MOUSE_di = []

for d in os.listdir(pwmdir_di):
    d = pwmdir_di + "/" + d
    if os.path.isdir(d): 
        os.chdir(d) 
        if d.split("_")[-1] == "HUMAN":
            files_pwm_HUMAN_di.append([f for f in os.listdir(os.getcwd()) if os.path.splitext(f)[1] == '.dpwm']) 
        if d.split("_")[-1] == "MOUSE":
            files_pwm_MOUSE_di.append([f for f in os.listdir(os.getcwd()) if os.path.splitext(f)[1] == '.dpwm'])
    os.chdir(pwmdir_di) 

print(len(files_pwm_HUMAN_di))
print(len(files_pwm_MOUSE_di))


def remove_duplicates(l):
    return list(set(l))

print("We have {N} TF before filtration of mono".format(N=len(remove_duplicates([x[0].split("~")[0].split("_")[0] for x in files_pwm_MOUSE_mono + files_pwm_HUMAN_mono]))))
print("We have {N} TF before filtration of di".format(N=len(remove_duplicates([x[0].split("~")[0].split("_")[0] for x in files_pwm_MOUSE_di + files_pwm_HUMAN_di]))))

In [ ]:
size_of_set = 10000
outfile_name_HUMAN_train = basicdir + "/all_mfa_file_HUMAN_" + str(size_of_set) + "_train.fasta"
outfile_name_MOUSE_train = basicdir + "/all_mfa_file_MOUSE_" + str(size_of_set) + "_train.fasta"
outfile_name_HUMAN_control = basicdir + "/all_mfa_file_HUMAN_" + str(size_of_set) + "_control.fasta"
outfile_name_MOUSE_control = basicdir + "/all_mfa_file_MOUSE_" + str(size_of_set) + "_control.fasta"

outfile_name_HUMAN_train_no_NF = outfile_name_HUMAN_train.split(".fasta")[0]+"_no_NF.fasta"
outfile_name_MOUSE_train_no_NF = outfile_name_MOUSE_train.split(".fasta")[0]+"_no_NF.fasta"
outfile_name_HUMAN_control_no_NF = outfile_name_HUMAN_control.split(".fasta")[0]+"_no_NF.fasta"
outfile_name_MOUSE_control_no_NF = outfile_name_MOUSE_control.split(".fasta")[0]+"_no_NF.fasta"

In [ ]:
headerStr_names_outfile_HUMAN_train = basicdir + "/headerStr_names_HUMAN_" + str(size_of_set) + "_train.csv"
headerStr_names_outfile_MOUSE_train = basicdir + "/headerStr_names_MOUSE_" + str(size_of_set) + "_train.csv"
headerStr_names_outfile_HUMAN_control = basicdir + "/headerStr_names_HUMAN_" + str(size_of_set) + "_control.csv"
headerStr_names_outfile_MOUSE_control = basicdir + "/headerStr_names_MOUSE_" + str(size_of_set) + "_control.csv"

In [ ]:
out_tab_file_HUMAN_train = basicdir + "/" + "out_tab_HUMAN_" + str(size_of_set) + "_train.tab"
out_tab_file_MOUSE_train = basicdir + "/" + "out_tab_MOUSE_" + str(size_of_set) + "_train.tab"
out_tab_file_HUMAN_control = basicdir + "/" + "out_tab_HUMAN_" + str(size_of_set) + "_control.tab"
out_tab_file_MOUSE_control = basicdir + "/" + "out_tab_MOUSE_" + str(size_of_set) + "_control.tab"

In [ ]:
trimmed = False
size = 20
begin_trim = size
end_trim = size
if trimmed == True:
    outfile_name_HUMAN_train_no_NF = outfile_name_HUMAN_train.split(".fasta")[0]+"_no_NF.fasta"
    outfile_name_MOUSE_train_no_NF = outfile_name_MOUSE_train.split(".fasta")[0]+"_no_NF.fasta"
    outfile_name_HUMAN_control_no_NF = outfile_name_HUMAN_control.split(".fasta")[0]+"_no_NF.fasta"
    outfile_name_MOUSE_control_no_NF = outfile_name_MOUSE_control.split(".fasta")[0]+"_no_NF.fasta"

    outfile_name_HUMAN_train_no_NF = outfile_name_HUMAN_train_no_NF.split(".fasta")[0] + "_trimmed_{begin_trim}_{end_trim}.fasta".format(begin_trim=begin_trim, end_trim=end_trim)
    outfile_name_MOUSE_train_no_NF = outfile_name_MOUSE_train_no_NF.split(".fasta")[0] + "_trimmed_{begin_trim}_{end_trim}.fasta".format(begin_trim=begin_trim, end_trim=end_trim)
    outfile_name_HUMAN_control_no_NF = outfile_name_HUMAN_control_no_NF.split(".fasta")[0] + "_trimmed_{begin_trim}_{end_trim}.fasta".format(begin_trim=begin_trim, end_trim=end_trim)
    outfile_name_MOUSE_control_no_NF = outfile_name_MOUSE_control_no_NF.split(".fasta")[0] + "_trimmed_{begin_trim}_{end_trim}.fasta".format(begin_trim=begin_trim, end_trim=end_trim)
else:
    outfile_name_HUMAN_train_no_NF = outfile_name_HUMAN_train.split(".fasta")[0]+"_no_NF.fasta"
    outfile_name_MOUSE_train_no_NF = outfile_name_MOUSE_train.split(".fasta")[0]+"_no_NF.fasta"
    outfile_name_HUMAN_control_no_NF = outfile_name_HUMAN_control.split(".fasta")[0]+"_no_NF.fasta"
    outfile_name_MOUSE_control_no_NF = outfile_name_MOUSE_control.split(".fasta")[0]+"_no_NF.fasta"

In [ ]:
with open(headerStr_names_outfile_HUMAN_train, "r") as output:
    headerStr_names_HUMAN_train = output.read().split('\n')
    print(headerStr_names_HUMAN_train[:10])

with open(headerStr_names_outfile_MOUSE_train, "r") as output:
    headerStr_names_MOUSE_train = output.read().split('\n')
    print(headerStr_names_MOUSE_train[:10])

with open(headerStr_names_outfile_HUMAN_control, "r") as output:
    headerStr_names_HUMAN_control = output.read().split('\n')
    print(headerStr_names_HUMAN_control[:10])

with open(headerStr_names_outfile_MOUSE_control, "r") as output:
    headerStr_names_MOUSE_control = output.read().split('\n')
    print(headerStr_names_MOUSE_control[:10])     

In [ ]:
family_annotation = pd.read_csv("./hocomoco11/core_annotation_HUMAN_mono.csv", sep='\t')

def find_TF_family_dict(family_annotation):
    TF_family_dict = defaultdict(None)
    for i, j in zip(family_annotation["Model"], family_annotation["TF family"]):
        if type(j) == str:
            TF_family_dict[i.split('_')[0]] = j.split('{')[1][:3]
    return TF_family_dict

TF_family_dict = find_TF_family_dict(family_annotation)

In [ ]:
out_tab_file_HUMAN_train = basicdir + "/" + "out_tab_HUMAN_" + str(size_of_set) + "_train.tab"
out_tab_file_MOUSE_train = basicdir + "/" + "out_tab_MOUSE_" + str(size_of_set) + "_train.tab"
out_tab_file_HUMAN_control = basicdir + "/" + "out_tab_HUMAN_" + str(size_of_set) + "_control.tab"
out_tab_file_MOUSE_control = basicdir + "/" + "out_tab_MOUSE_" + str(size_of_set) + "_control.tab"

In [ ]:
selected_HUMAN = ['ANDR_HUMAN',
 'AP2A_HUMAN',
 'CEBPB_HUMAN',
 'COE1_HUMAN',
 'CTCF_HUMAN',
 'E2F4_HUMAN',
 'ERG_HUMAN',
 'ESR1_HUMAN',
 'FLI1_HUMAN',
 'GATA1_HUMAN',
 'GATA2_HUMAN',
 'GATA3_HUMAN',
 'GCR_HUMAN',
 'HNF4A_HUMAN',
 'IRF1_HUMAN',
 'IRF4_HUMAN',
 'JUND_HUMAN',
 'MAFK_HUMAN',
 'MAX_HUMAN',
 'MYC_HUMAN',
 'P53_HUMAN',
 'PPARG_HUMAN',
 'PRGR_HUMAN',
 'REST_HUMAN',
 'RUNX1_HUMAN',
 'RXRA_HUMAN',
 'SOX2_HUMAN',
 'SPI1_HUMAN',
 'SRF_HUMAN',
 'STA5A_HUMAN',
 'STAT1_HUMAN',
 'STAT3_HUMAN',
 'TAL1_HUMAN',
 'TF65_HUMAN',
 'TFE2_HUMAN',
 'USF2_HUMAN']

selected_MOUSE = ['ANDR_MOUSE',
 'AP2A_MOUSE',
 'BATF_MOUSE',
 'CEBPB_MOUSE',
 'COE1_MOUSE',
 'CTCF_MOUSE',
 'E2F4_MOUSE',
 'ERG_MOUSE',
 'ESR1_MOUSE',
 'FLI1_MOUSE',
 'GATA1_MOUSE',
 'GATA2_MOUSE',
 'GATA3_MOUSE',
 'GCR_MOUSE',
 'HNF4A_MOUSE',
 'IRF1_MOUSE',
 'IRF4_MOUSE',
 'JUND_MOUSE',
 'MAFK_MOUSE',
 'MAX_MOUSE',
 'MYC_MOUSE',
 'MYOD1_MOUSE',
 'P53_MOUSE',
 'PPARG_MOUSE',
 'PRGR_MOUSE',
 'REST_MOUSE',
 'RUNX1_MOUSE',
 'RXRA_MOUSE',
 'SOX2_MOUSE',
 'SPI1_MOUSE',
 'SRF_MOUSE',
 'STA5A_MOUSE',
 'STAT1_MOUSE',
 'STAT3_MOUSE',
 'TAL1_MOUSE',
 'TF65_MOUSE',
 'TFE2_MOUSE',
 'USF2_MOUSE']

In [ ]:
for seq_org in ["HUMAN", "MOUSE"]:
    for pwm_org in ["HUMAN"]:
        N_features = 1000
        seed(42)
        model = "XGBClassifier"
        organism = "HUMAN_MOUSE"

        if pwm_org == "HUMAN":
            sel_TF = selected_HUMAN
        else:
            sel_TF = selected_MOUSE

        print(" ")
        сalc = 0
        for TF in sorted(selected_HUMAN): 
            print(TF)
            print("TF # {сalc} of {all_c}".format(all_c=len(selected_MOUSE), сalc=сalc))
            сalc += 1

            print("Dirname is:", TF)

            new_dir_name = outputdir + "/"+ TF.split("_")[0] + "/" + seq_org + "_seq_" + pwm_org + "_pwm_mono"
            print("New_dir_name", new_dir_name)
            print(" ")

            if not os.path.exists(new_dir_name):
                os.makedirs(new_dir_name)

            os.chdir(new_dir_name)
            dir_list = os.listdir(pwmdir_mono + "/" + TF)

            print("dir_list", len(dir_list))

            dir_list_short = [x.split("~")[2] for x in dir_list]

            try:
                pwm_benchmark = pd.read_csv("~/hocomoco11/curation/slices4bench_mono/out_sure.csv", sep='\t', header=None)
                wlogauc_datasets = pd.read_csv("/home/pavelkrav/hocomoco11/wlogauc/mono/" + TF.split("_")[1]+ "/out_sure.csv", sep='\t', header=None)
                wlogauc_datasets["names"] = [ x.split("~")[2].split(".")[0] for x in wlogauc_datasets[0]]
                wlogauc_datasets = wlogauc_datasets[wlogauc_datasets["names"] == TF]
                wlogauc_datasets = wlogauc_datasets.sort_values(1)
    
                pwm_benchmark["names"] = [ x.split(".")[0] for x in pwm_benchmark[0]]
                pwm_benchmark = pwm_benchmark[pwm_benchmark["names"] == TF]
                pwm_benchmark = pwm_benchmark[["macs" in x for x in pwm_benchmark[0].str.split(".")]]
                pwm_benchmark = [x+".pwm" for x in list(pwm_benchmark[0])]
                wlogauc_datasets = list(wlogauc_datasets[0])
                                 
                pwm_final_list = []
                
                for m in pwm_benchmark:
                    if m in dir_list_short:
                        pwm_final_list.append(dir_list[dir_list_short.index(m)])
                        
                print("Before: ", len(pwm_final_list), pwm_final_list)

            except FileNotFoundError:
                print("No file !!!!!")
                continue

            matrix_c = 0  

            if len(pwm_final_list) < 3:
                continue

            for matrix in range(len(pwm_final_list)):          
                matrix = pwm_final_list[matrix]
                print("PWM in process is:", matrix)
                matrix_dump = matrix

                if matrix_c < N_features:
                    print("Sarus is running ...")
                    print(" ")
                    print(matrix_c)
                    print(matrix_dump)
                    print(matrix_dump.split("~")[-1])
                    print(matrix_dump.split("~")[-1][:-3] + "pcm")
                    matrix_pcm = matrix_dump.split("~")[-1][:-3] + "pcm"
                    matrix_pcm_out = new_dir_name+"/"+matrix_pcm
                    print(matrix_pcm,matrix_pcm_out)
                    line = f"cp /home/pavelkrav/pcms_for_logo_mono/{matrix_pcm} {matrix_pcm_out}"
                    print(line)
                    p = subprocess.Popen(line, shell=True)
                    p.wait()

                    matrix_png_out = new_dir_name+"/"+matrix_dump.split("~")[-1][:-3] + "png"
                    line = f"ruby /home/ivavlakul/bin/chipmunk/pmflogo3.rb {matrix_pcm_out} {matrix_png_out}"
                    p = subprocess.Popen(line, shell=True)
                    p.wait()
                    if trimmed == True:

                        if seq_org == "MOUSE":
                            line = 'java -Xmx2G -cp /home/pavelkrav/sarus-2.0.2.jar ru.autosome.SARUS {mfa_file} {matrix} --skipn --show-non-matching --output-scoring-mode score besthit | grep -v \> > {out_file}'.format(sarsus_home = root + "/sarus/releases/sarus-2.0.1.jar",
                            mfa_file = outfile_name_MOUSE_train_no_NF,
                            matrix = '/'.join([pwmdir_mono, TF, matrix]),
                            out_file = '/'.join([new_dir_name, str(matrix_c) + "_" + "_".join(matrix.split(".")[:-1]) + "_feature_compare_table_train_trimmed_{begin_trim}_{end_trim}.tab".format(begin_trim=begin_trim, end_trim=end_trim)]))


                        if seq_org == "HUMAN":
                            line = 'java -Xmx2G -cp /home/pavelkrav/sarus-2.0.2.jar ru.autosome.SARUS {mfa_file} {matrix} --skipn --show-non-matching --output-scoring-mode score besthit | grep -v \> > {out_file}'.format(sarsus_home = root + "/sarus/releases/sarus-2.0.1.jar",
                            mfa_file = outfile_name_HUMAN_train_no_NF,
                            matrix = '/'.join([pwmdir_mono, TF, matrix]),
                            out_file = '/'.join([new_dir_name, str(matrix_c) + "_" + "_".join(matrix.split(".")[:-1]) + "_feature_compare_table_train_trimmed_{begin_trim}_{end_trim}.tab".format(begin_trim=begin_trim, end_trim=end_trim)]))

                            
                        p = subprocess.Popen(line, shell=True)
                        p.wait()

                        !rm tmp.sh

                        line1 = "awk -F. \'{print $1\".\"substr($2,1,2)}\' " + str(matrix_c) + "_" + "_".join(matrix.split(".")[:-1]) + "_feature_compare_table_train_trimmed_{begin_trim}_{end_trim}.tab".format(begin_trim=begin_trim, end_trim=end_trim) + " > " + str(matrix_c) + "_" + "_".join(matrix.split(".")[:-1]) + "_feature_compare_table_train_cut_trimmed_{begin_trim}_{end_trim}.tab".format(begin_trim=begin_trim, end_trim=end_trim)
                        line2 = "rm " + str(matrix_c) + "_" + "_".join(matrix.split(".")[:-1]) + "_feature_compare_table_train_trimmed_{begin_trim}_{end_trim}.tab".format(begin_trim=begin_trim, end_trim=end_trim)

                        with open("tmp.sh", "w") as f:
                            f.write(line1)
                            f.write(" ; ")
                            f.write(line2)
                        !bash tmp.sh

                        print("Sarus is running ...")
                        print(" ")
                        if seq_org == "MOUSE":
                            line = 'java -Xmx2G -cp /home/pavelkrav/sarus-2.0.2.jar ru.autosome.SARUS {mfa_file} {matrix} --skipn --show-non-matching --output-scoring-mode score besthit | grep -v \> > {out_file}'.format(sarsus_home = root + "/sarus/releases/sarus-2.0.1.jar",
                            mfa_file = outfile_name_MOUSE_control_no_NF,
                            matrix = '/'.join([pwmdir_mono, TF, matrix]),
                            out_file = '/'.join([new_dir_name, str(matrix_c) + "_" + "_".join(matrix.split(".")[:-1]) + "_feature_compare_table_control_trimmed_{begin_trim}_{end_trim}.tab".format(begin_trim=begin_trim, end_trim=end_trim)]))


                        if seq_org == "HUMAN":
                            line = 'java -Xmx2G -cp /home/pavelkrav/sarus-2.0.2.jar ru.autosome.SARUS {mfa_file} {matrix} --skipn --show-non-matching --output-scoring-mode score besthit | grep -v \> > {out_file}'.format(sarsus_home = root + "/sarus/releases/sarus-2.0.1.jar",
                            mfa_file = outfile_name_HUMAN_control_no_NF,
                            matrix = '/'.join([pwmdir_mono, TF, matrix]),
                            out_file = '/'.join([new_dir_name, str(matrix_c) + "_" + "_".join(matrix.split(".")[:-1]) + "_feature_compare_table_control_trimmed_{begin_trim}_{end_trim}.tab".format(begin_trim=begin_trim, end_trim=end_trim)]))

                        p = subprocess.Popen(line, shell=True)
                        p.wait()

                        !rm tmp.sh
                        
                        line1 = "awk -F. \'{print $1\".\"substr($2,1,2)}\' " + str(matrix_c) + "_" + "_".join(matrix.split(".")[:-1]) + "_feature_compare_table_control_trimmed_{begin_trim}_{end_trim}.tab".format(begin_trim=begin_trim, end_trim=end_trim) + " > " + str(matrix_c) + "_" + "_".join(matrix.split(".")[:-1]) + "_feature_compare_table_control_cut_trimmed_{begin_trim}_{end_trim}.tab".format(begin_trim=begin_trim, end_trim=end_trim)

                        line2 = "rm " + str(matrix_c) + "_" + "_".join(matrix.split(".")[:-1]) + "_feature_compare_table_control_trimmed_{begin_trim}_{end_trim}.tab".format(begin_trim=begin_trim, end_trim=end_trim)
                        with open("tmp.sh", "w") as f:
                            f.write(line1)
                            f.write(" ; ")
                            f.write(line2)
                        !bash tmp.sh
                    else:
                        if seq_org == "MOUSE":
                            line = 'java -Xmx2G -cp /home/pavelkrav/sarus-2.0.2.jar ru.autosome.SARUS {mfa_file} {matrix} --skipn --show-non-matching --output-scoring-mode score besthit | grep -v \> > {out_file}'.format(sarsus_home = root + "/sarus/releases/sarus-2.0.1.jar",
                            mfa_file = outfile_name_MOUSE_train_no_NF,
                            matrix = '/'.join([pwmdir_mono, TF, matrix]),
                            out_file = '/'.join([new_dir_name, str(matrix_c) + "_" + "_".join(matrix.split(".")[:-1]) + "_feature_compare_table_train.tab"]))

                        if seq_org == "HUMAN":
                            line = 'java -Xmx2G -cp /home/pavelkrav/sarus-2.0.2.jar ru.autosome.SARUS {mfa_file} {matrix} --skipn --show-non-matching --output-scoring-mode score besthit | grep -v \> > {out_file}'.format(sarsus_home = root + "/sarus/releases/sarus-2.0.1.jar",
                            mfa_file = outfile_name_HUMAN_train_no_NF,
                            matrix = '/'.join([pwmdir_mono, TF, matrix]),
                            out_file = '/'.join([new_dir_name, str(matrix_c) + "_" + "_".join(matrix.split(".")[:-1]) + "_feature_compare_table_train.tab"]))

                        p = subprocess.Popen(line, shell=True)
                        p.wait()

                        !rm tmp.sh

                        line1 = "awk -F. \'{print $1\".\"substr($2,1,2)}\' " + str(matrix_c) + "_" + "_".join(matrix.split(".")[:-1]) + "_feature_compare_table_train.tab" + " > " + str(matrix_c) + "_" + "_".join(matrix.split(".")[:-1]) + "_feature_compare_table_train_cut.tab"
                        line2 = "rm " + str(matrix_c) + "_" + "_".join(matrix.split(".")[:-1]) + "_feature_compare_table_train.tab"

                        with open("tmp.sh", "w") as f:
                            f.write(line1)
                            f.write(" ; ")
                            f.write(line2)
                        !bash tmp.sh

                        print("Sarus is running ...")
                        print(" ")
                        if seq_org == "MOUSE":
                            line = 'java -Xmx2G -cp /home/pavelkrav/sarus-2.0.2.jar ru.autosome.SARUS {mfa_file} {matrix} --skipn --show-non-matching --output-scoring-mode score besthit | grep -v \> > {out_file}'.format(sarsus_home = root + "/sarus/releases/sarus-2.0.1.jar",
                            mfa_file = outfile_name_MOUSE_control_no_NF,
                            matrix = '/'.join([pwmdir_mono, TF, matrix]),
                            out_file = '/'.join([new_dir_name, str(matrix_c) + "_" + "_".join(matrix.split(".")[:-1]) + "_feature_compare_table_control.tab"]))
                            #print(line)

                        if seq_org == "HUMAN":
                            line = 'java -Xmx2G -cp /home/pavelkrav/sarus-2.0.2.jar ru.autosome.SARUS {mfa_file} {matrix} --skipn --show-non-matching --output-scoring-mode score besthit | grep -v \> > {out_file}'.format(sarsus_home = root + "/sarus/releases/sarus-2.0.1.jar",
                            mfa_file = outfile_name_HUMAN_control_no_NF,
                            matrix = '/'.join([pwmdir_mono, TF, matrix]),
                            out_file = '/'.join([new_dir_name, str(matrix_c) + "_" + "_".join(matrix.split(".")[:-1]) + "_feature_compare_table_control.tab"]))

                        p = subprocess.Popen(line, shell=True)
                        p.wait()

                        !rm tmp.sh

                        line1 = "awk -F. \'{print $1\".\"substr($2,1,2)}\' " + str(matrix_c) + "_" + "_".join(matrix.split(".")[:-1]) + "_feature_compare_table_control.tab" + " > " + str(matrix_c) + "_" + "_".join(matrix.split(".")[:-1]) + "_feature_compare_table_control_cut.tab"

                        line2 = "rm " + str(matrix_c) + "_" + "_".join(matrix.split(".")[:-1]) + "_feature_compare_table_control.tab"
                        with open("tmp.sh", "w") as f:
                            f.write(line1)
                            f.write(" ; ")
                            f.write(line2)
                        !bash tmp.sh

                    pwm_name = '/'.join([pwmdir_mono, TF, matrix])
                    p = subprocess.Popen("cp {pwm_name} {new_dir_name}".format(pwm_name=pwm_name, new_dir_name=new_dir_name + "/" + str(matrix_c) + ".pwm"), shell=True)
                    p.wait()

                else:
                    break
                matrix_c += 1

            p = subprocess.Popen("tar -cvzf pwm_files_{TF}.tar.gz *.pwm".format(TF=TF), shell=True) 
            p.wait()
            os.chdir(outputdir)


In [ ]:
for seq_org in ["HUMAN", "MOUSE"]:
    for pwm_org in ["HUMAN"]:

        N_features = 1000
        seed(42)
        model = "XGBClassifier"
        organism = "HUMAN_MOUSE"

        if pwm_org == "HUMAN":
            sel_TF = selected_HUMAN
        else:
            sel_TF = selected_MOUSE

        print(" ")
        сalc = 0
        for TF in sorted(selected_HUMAN): 
            print(TF)
            print("TF # {сalc} of {all_c}".format(all_c=len(selected_MOUSE), сalc=сalc))
            сalc += 1

            print("Dirname is:", TF)

            new_dir_name = outputdir + "/"+ TF.split("_")[0] + "/" + seq_org + "_seq_" + pwm_org + "_pwm_di"
            print("New_dir_name", new_dir_name)
            print(" ")

            
            if not os.path.exists(new_dir_name):
                os.makedirs(new_dir_name)
            else:
                p = subprocess.Popen(f"rm -rf {new_dir_name}", shell=True)
                p.wait()
    
                os.makedirs(new_dir_name)

            os.chdir(new_dir_name)
            dir_list = os.listdir(pwmdir_di + "/" + TF) 

            print("dir_list", len(dir_list))

            dir_list_short = [x.split("~")[2] for x in dir_list]

            try:
                pwm_benchmark = pd.read_csv("~/hocomoco11/curation/slices4bench_di/out_sure.csv", sep='\t', header=None) 
                wlogauc_datasets = pd.read_csv("/home/pavelkrav/hocomoco11/wlogauc/di/" + TF.split("_")[1]+ "/out_sure.csv", sep='\t', header=None) 
                wlogauc_datasets["names"] = [ x.split("~")[2].split(".")[0] for x in wlogauc_datasets[0]]
                wlogauc_datasets = wlogauc_datasets[wlogauc_datasets["names"] == TF]
                wlogauc_datasets = wlogauc_datasets.sort_values(1)
    
                pwm_benchmark["names"] = [ x.split(".")[0] for x in pwm_benchmark[0]]
                pwm_benchmark = pwm_benchmark[pwm_benchmark["names"] == TF]
                pwm_benchmark = pwm_benchmark[["macs" in x for x in pwm_benchmark[0].str.split(".")]]
                pwm_benchmark = [x+".dpwm" for x in list(pwm_benchmark[0])]
                wlogauc_datasets = list(wlogauc_datasets[0])
                                 
                pwm_final_list = []
                
                for m in pwm_benchmark: 
                    if m in dir_list_short:
                        pwm_final_list.append(dir_list[dir_list_short.index(m)])
                        
                print("Before: ", len(pwm_final_list), pwm_final_list)

            except FileNotFoundError:
                print("No file !!!!!")
                continue

            matrix_c = 0

            if len(pwm_final_list) < 3:
                continue

            for matrix in range(len(pwm_final_list)):              
                matrix = pwm_final_list[matrix]
                print("PWM in process is:", matrix)
                matrix_dump = matrix

                if matrix_c < N_features:
                    print("Sarus is running ...") 
                    print(" ")
                    print(matrix_dump)
                    print(matrix_dump.split("~")[-1])
                    print(matrix_dump.split("~")[-1][:-3] + "pcm")
                    matrix_pcm = matrix_dump.split("~")[-1][:-3] + "pcm"
                    matrix_pcm_out = new_dir_name+"/"+matrix_pcm
                    print(matrix_pcm,matrix_pcm_out)
                    line = f"cp /home/pavelkrav/pcms_for_logo_di/{matrix_pcm} {matrix_pcm_out}"
                    print(line)
                    p = subprocess.Popen(line, shell=True)
                    p.wait()

                    matrix_pcm_out2 = new_dir_name+"/"+matrix_dump.split("~")[-1][:-4] + "_2.dpcm"
                    matrix_png_out = new_dir_name+"/"+matrix_dump.split("~")[-1][:-4] + "png"
                    line = f"tail -n +2 {matrix_pcm_out} > {matrix_pcm_out2}; ruby /home/ivavlakul/bin/chipmunk/dpmflogo3_mod.rb {matrix_pcm_out2} {matrix_png_out}"
                    print(line)
                    p = subprocess.Popen(line, shell=True)
                    p.wait()
                    if trimmed == True:

                        if seq_org == "MOUSE":
                            line = 'java -Xmx2G -cp {sarsus_home} ru.autosome.di.SARUS {mfa_file} {matrix} --skipn --show-non-matching --output-scoring-mode score besthit | grep -v \> > {out_file}'.format(sarsus_home = root + "/sarus/releases/sarus-2.0.1.jar",
                            mfa_file = outfile_name_MOUSE_train_no_NF,
                            matrix = '/'.join([pwmdir_di, TF, matrix]),
                            out_file = '/'.join([new_dir_name, str(matrix_c) + "_" + "_".join(matrix.split(".")[:-1]) + "_feature_compare_table_train_trimmed_{begin_trim}_{end_trim}.tab".format(begin_trim=begin_trim, end_trim=end_trim)]))

                        if seq_org == "HUMAN":
                            line = 'java -Xmx2G -cp {sarsus_home} ru.autosome.di.SARUS {mfa_file} {matrix} --skipn --show-non-matching --output-scoring-mode score besthit | grep -v \> > {out_file}'.format(sarsus_home = root + "/sarus/releases/sarus-2.0.1.jar",
                            mfa_file = outfile_name_HUMAN_train_no_NF,
                            matrix = '/'.join([pwmdir_di, TF, matrix]),
                            out_file = '/'.join([new_dir_name, str(matrix_c) + "_" + "_".join(matrix.split(".")[:-1]) + "_feature_compare_table_train_trimmed_{begin_trim}_{end_trim}.tab".format(begin_trim=begin_trim, end_trim=end_trim)]))
                            
                        p = subprocess.Popen(line, shell=True)
                        p.wait()

                        !rm tmp.sh

                        line1 = "awk -F. \'{print $1\".\"substr($2,1,2)}\' " + str(matrix_c) + "_" + "_".join(matrix.split(".")[:-1]) + "_feature_compare_table_train_trimmed_{begin_trim}_{end_trim}.tab".format(begin_trim=begin_trim, end_trim=end_trim) + " > " + str(matrix_c) + "_" + "_".join(matrix.split(".")[:-1]) + "_feature_compare_table_train_cut_trimmed_{begin_trim}_{end_trim}.tab".format(begin_trim=begin_trim, end_trim=end_trim)
                        line2 = "rm " + str(matrix_c) + "_" + "_".join(matrix.split(".")[:-1]) + "_feature_compare_table_train_trimmed_{begin_trim}_{end_trim}.tab".format(begin_trim=begin_trim, end_trim=end_trim)

                        with open("tmp.sh", "w") as f:
                            f.write(line1)
                            f.write(" ; ")
                            f.write(line2)
                        !bash tmp.sh


                        print(" ")
                        if seq_org == "MOUSE":
                            line = 'java -Xmx2G -cp {sarsus_home} ru.autosome.di.SARUS {mfa_file} {matrix} --skipn --show-non-matching --output-scoring-mode score besthit | grep -v \> > {out_file}'.format(sarsus_home = root + "/sarus/releases/sarus-2.0.1.jar",
                            mfa_file = outfile_name_MOUSE_control_no_NF,
                            matrix = '/'.join([pwmdir_di, TF, matrix]),
                            out_file = '/'.join([new_dir_name, str(matrix_c) + "_" + "_".join(matrix.split(".")[:-1]) + "_feature_compare_table_control_trimmed_{begin_trim}_{end_trim}.tab".format(begin_trim=begin_trim, end_trim=end_trim)]))
                            #print(line)

                        if seq_org == "HUMAN":
                            line = 'java -Xmx2G -cp {sarsus_home} ru.autosome.di.SARUS {mfa_file} {matrix} --skipn --show-non-matching --output-scoring-mode score besthit | grep -v \> > {out_file}'.format(sarsus_home = root + "/sarus/releases/sarus-2.0.1.jar",
                            mfa_file = outfile_name_HUMAN_control_no_NF,
                            matrix = '/'.join([pwmdir_di, TF, matrix]),
                            out_file = '/'.join([new_dir_name, str(matrix_c) + "_" + "_".join(matrix.split(".")[:-1]) + "_feature_compare_table_control_trimmed_{begin_trim}_{end_trim}.tab".format(begin_trim=begin_trim, end_trim=end_trim)]))

                        p = subprocess.Popen(line, shell=True)
                        p.wait()

                        !rm tmp.sh

                        line1 = "awk -F. \'{print $1\".\"substr($2,1,2)}\' " + str(matrix_c) + "_" + "_".join(matrix.split(".")[:-1]) + "_feature_compare_table_control_trimmed_{begin_trim}_{end_trim}.tab".format(begin_trim=begin_trim, end_trim=end_trim) + " > " + str(matrix_c) + "_" + "_".join(matrix.split(".")[:-1]) + "_feature_compare_table_control_cut_trimmed_{begin_trim}_{end_trim}.tab".format(begin_trim=begin_trim, end_trim=end_trim)

                        line2 = "rm " + str(matrix_c) + "_" + "_".join(matrix.split(".")[:-1]) + "_feature_compare_table_control_trimmed_{begin_trim}_{end_trim}.tab".format(begin_trim=begin_trim, end_trim=end_trim)
                        with open("tmp.sh", "w") as f:
                            f.write(line1)
                            f.write(" ; ")
                            f.write(line2)
                        !bash tmp.sh
                    else:
                        if seq_org == "MOUSE":
                            line = 'java -Xmx2G -cp {sarsus_home} ru.autosome.di.SARUS {mfa_file} {matrix} --skipn --show-non-matching --output-scoring-mode score besthit | grep -v \> > {out_file}'.format(sarsus_home = root + "/sarus/releases/sarus-2.0.1.jar",
                            mfa_file = outfile_name_MOUSE_train_no_NF,
                            matrix = '/'.join([pwmdir_di, TF, matrix]),
                            out_file = '/'.join([new_dir_name, str(matrix_c) + "_" + "_".join(matrix.split(".")[:-1]) + "_feature_compare_table_train.tab"]))

                        if seq_org == "HUMAN":
                            line = 'java -Xmx2G -cp {sarsus_home} ru.autosome.di.SARUS {mfa_file} {matrix} --skipn --show-non-matching --output-scoring-mode score besthit | grep -v \> > {out_file}'.format(sarsus_home = root + "/sarus/releases/sarus-2.0.1.jar",
                            mfa_file = outfile_name_HUMAN_train_no_NF,
                            matrix = '/'.join([pwmdir_di, TF, matrix]),
                            out_file = '/'.join([new_dir_name, str(matrix_c) + "_" + "_".join(matrix.split(".")[:-1]) + "_feature_compare_table_train.tab"]))

                        p = subprocess.Popen(line, shell=True)
                        p.wait()

                        !rm tmp.sh

                        line1 = "awk -F. \'{print $1\".\"substr($2,1,2)}\' " + str(matrix_c) + "_" + "_".join(matrix.split(".")[:-1]) + "_feature_compare_table_train.tab" + " > " + str(matrix_c) + "_" + "_".join(matrix.split(".")[:-1]) + "_feature_compare_table_train_cut.tab"
                        line2 = "rm " + str(matrix_c) + "_" + "_".join(matrix.split(".")[:-1]) + "_feature_compare_table_train.tab"

                        with open("tmp.sh", "w") as f:
                            f.write(line1)
                            f.write(" ; ")
                            f.write(line2)
                        !bash tmp.sh

                        print("Sarus is running ...") 
                        print(" ")
                        if seq_org == "MOUSE":
                            line = 'java -Xmx2G -cp {sarsus_home} ru.autosome.di.SARUS {mfa_file} {matrix} --skipn --show-non-matching --output-scoring-mode score besthit | grep -v \> > {out_file}'.format(sarsus_home = root + "/sarus/releases/sarus-2.0.1.jar",
                            mfa_file = outfile_name_MOUSE_control_no_NF,
                            matrix = '/'.join([pwmdir_di, TF, matrix]),
                            out_file = '/'.join([new_dir_name, str(matrix_c) + "_" + "_".join(matrix.split(".")[:-1]) + "_feature_compare_table_control.tab"]))
                            #print(line)

                        if seq_org == "HUMAN":
                            line = 'java -Xmx2G -cp {sarsus_home} ru.autosome.di.SARUS {mfa_file} {matrix} --skipn --show-non-matching --output-scoring-mode score besthit | grep -v \> > {out_file}'.format(sarsus_home = root + "/sarus/releases/sarus-2.0.1.jar",
                            mfa_file = outfile_name_HUMAN_control_no_NF,
                            matrix = '/'.join([pwmdir_di, TF, matrix]),
                            out_file = '/'.join([new_dir_name, str(matrix_c) + "_" + "_".join(matrix.split(".")[:-1]) + "_feature_compare_table_control.tab"]))

                        p = subprocess.Popen(line, shell=True)
                        p.wait()

                        !rm tmp.sh

                        line1 = "awk -F. \'{print $1\".\"substr($2,1,2)}\' " + str(matrix_c) + "_" + "_".join(matrix.split(".")[:-1]) + "_feature_compare_table_control.tab" + " > " + str(matrix_c) + "_" + "_".join(matrix.split(".")[:-1]) + "_feature_compare_table_control_cut.tab"

                        line2 = "rm " + str(matrix_c) + "_" + "_".join(matrix.split(".")[:-1]) + "_feature_compare_table_control.tab"
                        with open("tmp.sh", "w") as f:
                            f.write(line1)
                            f.write(" ; ")
                            f.write(line2)
                        !bash tmp.sh

                    pwm_name = '/'.join([pwmdir_di, TF, matrix])
                    p = subprocess.Popen("cp {pwm_name} {new_dir_name}".format(pwm_name=pwm_name, new_dir_name=new_dir_name + "/" + str(matrix_c) + ".dpwm"), shell=True)
                    p.wait()

                else:
                    break
                matrix_c += 1

            p = subprocess.Popen("tar -cvzf pwm_files_{TF}.tar.gz *.dpwm".format(TF=TF), shell=True)
            p.wait()
            os.chdir(outputdir)